# Global Analysis of All Reconstructed Matrices

In [306]:
from pathlib import Path
import warnings
import networkx as nx
import json
import numpy as np
import pandas as pd
import torch

np.set_printoptions(suppress=True, precision=4)
warnings.filterwarnings('ignore', message='.*Clustering large matrix.*')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

Device: cuda


# Step 1: Load Full Reconstructed Dataset (Local PC)
Load reconstructed matrices.

In [307]:
WINDOW_LENGTH = 252
STRIDE = 5
FORWARD_DAYS = 21
FILE_NAME = 'data_00_20'
DATASET_NAME = f'{FILE_NAME}_w{WINDOW_LENGTH}_s{STRIDE}'
MODEL_TYPE = 'AE'  # 'PCA', 'AE', 'linearAE', 'VAE'
RUN_NAME = 'AE_100dim_0001'

project_root = Path.cwd().resolve().parent
model_root = project_root / 'models' / DATASET_NAME / f'{FORWARD_DAYS}_days_gap' / MODEL_TYPE / RUN_NAME

use_analysis_outputs = MODEL_TYPE in {'linearAE', 'AE', 'PCA', 'VAE'}
recon_dir = model_root / 'analysis_outputs' if use_analysis_outputs else model_root

DATASET_ORDER = ['train', 'val', 'test', 'all']
DATASET_FILES = {
    name: recon_dir / f'{name}_reconstructed_{RUN_NAME}.pt'
    for name in DATASET_ORDER
}
print(f'Model root: {model_root}')
RESULTS_ROOT = project_root / 'results' / DATASET_NAME / f'{FORWARD_DAYS}_days_gap'
RESULTS_DIRS = {
    name: RESULTS_ROOT / f'reconstruction_analysis_{name}' / RUN_NAME
    for name in DATASET_ORDER
}

for path in RESULTS_DIRS.values():
    path.mkdir(parents=True, exist_ok=True)

#missing = [name for name, path in DATASET_FILES.items() if not path.exists()]
#if missing:
    #missing_list = ', '.join(missing)
    #raise FileNotFoundError(f'Missing reconstructed files for: {missing_list}')

print(f'Dataset selected: {DATASET_NAME}')
print(f'Model type: {MODEL_TYPE} | Run: {RUN_NAME}')
print('Reconstructed files:')
for name, path in DATASET_FILES.items():
    print(f'  {name}: {path.name}')

Model root: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\models\data_00_20_w252_s5\21_days_gap\AE\AE_100dim_0001
Dataset selected: data_00_20_w252_s5
Model type: AE | Run: AE_100dim_0001
Reconstructed files:
  train: train_reconstructed_AE_100dim_0001.pt
  val: val_reconstructed_AE_100dim_0001.pt
  test: test_reconstructed_AE_100dim_0001.pt
  all: all_reconstructed_AE_100dim_0001.pt


In [308]:
def load_corr_payload(pt_path: Path):
    payload = torch.load(pt_path, map_location='cpu')
    corr_tensor = payload.get('corr_tensor', None)
    recon_tensor = payload.get('corr_tensor_reconstructed', None)
    indices = payload.get('indices', None)
    meta = {k: v for k, v in payload.items() if k != 'corr_tensor'}

    if corr_tensor is None:
        raise KeyError('corr_tensor key not found in .pt file')
    if recon_tensor is None:
        raise KeyError('corr_tensor_reconstructed key not found in .pt file')
    if corr_tensor.ndim != 3 or corr_tensor.shape[1] != corr_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor: {corr_tensor.shape}')
    if recon_tensor.ndim != 3 or recon_tensor.shape[1] != recon_tensor.shape[2]:
        raise ValueError(f'Invalid shape for corr_tensor_reconstructed: {recon_tensor.shape}')

    return corr_tensor.float(), recon_tensor.float(), indices, meta


print('Ready to analyze reconstructed datasets:')
for name, path in DATASET_FILES.items():
    print(f'  {name}: {path.name}')

Ready to analyze reconstructed datasets:
  train: train_reconstructed_AE_100dim_0001.pt
  val: val_reconstructed_AE_100dim_0001.pt
  test: test_reconstructed_AE_100dim_0001.pt
  all: all_reconstructed_AE_100dim_0001.pt


# Errors Statistics (MSE,MAE,Frobenius on full Reconstructed Dataset)

In [309]:
def reconstruction_errors(original: np.ndarray, reconstructed: np.ndarray):
    diff = original - reconstructed

    mse_per_matrix = np.mean(diff ** 2, axis=(1, 2))
    mae_per_matrix = np.mean(np.abs(diff), axis=(1, 2))
    fro_per_matrix = np.linalg.norm(diff, ord='fro', axis=(1, 2))

    summary = pd.DataFrame({
        'MSE': mse_per_matrix,
        'MAE': mae_per_matrix,
        'Frobenius': fro_per_matrix,
    })

    stats = summary.agg(['mean', 'std', 'min', 'median', 'max']).T
    return summary, stats

In [310]:
def compute_errors_payload(gt_corr: torch.Tensor, recon_corr: torch.Tensor):
    gt_corr_np = gt_corr.cpu().numpy()
    recon_corr_np = recon_corr.cpu().numpy()

    errors_df, errors_stats = reconstruction_errors(gt_corr_np, recon_corr_np)
    errors_stats_dict = errors_stats.T.to_dict()

    return gt_corr_np, recon_corr_np, errors_df, errors_stats, errors_stats_dict

In [311]:
def mst_edge_set(mst) -> set:
    return {frozenset(edge) for edge in mst.edges()}

def degree_distribution(mst, n_assets: int) -> np.ndarray:
    degrees = np.array([deg for _, deg in mst.degree()], dtype=int)
    counts = np.bincount(degrees, minlength=n_assets)
    return counts / counts.sum()

def compare_mst_metrics(mst_orig, mst_recon, n_assets: int, top_k: int):
    # Edges overlap and Jaccard
    edges_orig = mst_edge_set(mst_orig)
    edges_recon = mst_edge_set(mst_recon)
    common_edges = len(edges_orig & edges_recon)
    total_edges = max(n_assets - 1, 1)
    edge_overlap_pct = 100.0 * common_edges / total_edges
    edge_jaccard_pct = 100.0 * common_edges / max(len(edges_orig | edges_recon), 1)

    # Degree distribution and L1 distance
    dist_orig = degree_distribution(mst_orig, n_assets)
    dist_recon = degree_distribution(mst_recon, n_assets)
    degree_l1 = float(np.sum(np.abs(dist_orig - dist_recon)))

    # Average path length (weighted by distance)
    avg_path_len = nx.average_shortest_path_length(mst_orig, weight='weight')
    avg_path_len_recon = nx.average_shortest_path_length(mst_recon, weight='weight')
    avg_path_len_diff = float(abs(avg_path_len - avg_path_len_recon))

    # Average path length (unweighted, treating all edges as length 1)
    avg_path_len_unw = nx.average_shortest_path_length(mst_orig)
    avg_path_len_unw_recon = nx.average_shortest_path_length(mst_recon)
    avg_path_len_unw_diff = float(abs(avg_path_len_unw - avg_path_len_unw_recon))

    # Betweenness centrality top-k overlap
    top_k = int(min(top_k, n_assets))
    bet_orig = nx.betweenness_centrality(mst_orig, weight='weight', normalized=True)
    bet_recon = nx.betweenness_centrality(mst_recon, weight='weight', normalized=True)
    top_orig = {n for n, _ in sorted(bet_orig.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    top_recon = {n for n, _ in sorted(bet_recon.items(), key=lambda kv: kv[1], reverse=True)[:top_k]}
    bet_overlap_pct = 100.0 * len(top_orig & top_recon) / max(top_k, 1)

    return {
        'edge_overlap_pct': edge_overlap_pct,
        'edge_jaccard_pct': edge_jaccard_pct,
        'degree_l1': degree_l1,
        'avg_path_len': float(avg_path_len),
        'avg_path_len_recon': float(avg_path_len_recon),
        'avg_path_len_diff': avg_path_len_diff,
        'avg_path_len_unweighted': float(avg_path_len_unw),
        'avg_path_len_unweighted_recon': float(avg_path_len_unw_recon),
        'avg_path_len_unweighted_diff': avg_path_len_unw_diff,
        'betweenness_topk_overlap_pct': bet_overlap_pct,
    }

In [312]:
def sanitize_correlation_matrix(corr_matrix):
    corr_matrix = np.clip(corr_matrix, -1.0, 1.0)
    corr_matrix = (corr_matrix + corr_matrix.T) / 2.0
    np.fill_diagonal(corr_matrix, 1.0)
    
    return corr_matrix

def build_distance_matrix(corr_matrix):
    dist_matrix = np.sqrt(np.maximum(0.0, 2.0 * (1.0 - corr_matrix)))
    dist_matrix = (dist_matrix + dist_matrix.T) / 2.0
    np.fill_diagonal(dist_matrix, 0.0)
    
    return dist_matrix

def corr_to_mst(corr_matrix):
    """
    Converte una matrice di correlazione in un oggetto MST di NetworkX.
    Usa la metrica di distanza d = sqrt(2 * (1 - rho))
    """
    # 1. Calcolo della matrice delle distanze
    corr_matrix = sanitize_correlation_matrix(corr_matrix)
    dist_matrix = build_distance_matrix(corr_matrix)
    
    # 2. Creazione del grafo completo
    G = nx.from_numpy_array(dist_matrix)
    
    # 3. Calcolo del Minimum Spanning Tree
    mst = nx.minimum_spanning_tree(G, weight='weight')
    
    return mst

In [313]:
all_datasets_metrics = {}

for dataset_name, file_path in DATASET_FILES.items():
    print(f"\n=== Processing {dataset_name} ===")

    gt_corr, recon_corr, indices, meta = load_corr_payload(file_path)
    gt_corr_np, recon_corr_np, _, errors_stats, errors_stats_dict = compute_errors_payload(
        gt_corr,
        recon_corr,
    )

    print(f'Reconstruction error statistics ({dataset_name} dataset):')
    display(errors_stats)

    n_samples = gt_corr_np.shape[0]
    n_assets = gt_corr_np.shape[1]
    top_k_centrality = 10

    print(f"Starting MST analysis for {n_samples} matrices...")

    all_metrics = []
    for i in range(n_samples):
        curr_gt = gt_corr_np[i]
        curr_recon = recon_corr_np[i]

        mst_gt = corr_to_mst(curr_gt)
        mst_recon = corr_to_mst(curr_recon)

        metrics = compare_mst_metrics(
            mst_gt,
            mst_recon,
            n_assets=n_assets,
            top_k=top_k_centrality,
        )

        all_metrics.append(metrics)

        if (i + 1) % 10 == 0:
            print(f"Processed {i + 1}/{n_samples} matrices...")

    results_df = pd.DataFrame(all_metrics)
    stats_df = results_df.describe().T
    stats_dict = stats_df.to_dict(orient='index')

    combined_metrics = {
        'reconstruction_errors': errors_stats_dict,
        'mst_metrics': stats_dict,
    }

    output_path = RESULTS_DIRS[dataset_name] / f'reconstruction_errors_{dataset_name}.json'
    with open(output_path, 'w') as f:
        json.dump(combined_metrics, f, indent=4)

    print(f"Saved metrics to: {output_path}")
    print(f"\n--- Summary Statistics ({dataset_name} dataset) ---")
    display(stats_df)

    all_datasets_metrics[dataset_name] = {
        'errors_stats': errors_stats,
        'mst_stats': stats_df,
    }


=== Processing train ===
Reconstruction error statistics (train dataset):


,mean,std,min,median,max
MSE,0.007415,0.001419,0.004762,0.007429,0.012107
MAE,0.068010,0.006512,0.054817,0.068169,0.086575
Frobenius,8.571971,0.820633,6.900535,8.619228,11.003367


Starting MST analysis for 679 matrices...
Processed 10/679 matrices...
Processed 20/679 matrices...
Processed 30/679 matrices...
Processed 40/679 matrices...
Processed 50/679 matrices...
Processed 60/679 matrices...
Processed 70/679 matrices...
Processed 80/679 matrices...
Processed 90/679 matrices...
Processed 100/679 matrices...
Processed 110/679 matrices...
Processed 120/679 matrices...
Processed 130/679 matrices...
Processed 140/679 matrices...
Processed 150/679 matrices...
Processed 160/679 matrices...
Processed 170/679 matrices...
Processed 180/679 matrices...
Processed 190/679 matrices...
Processed 200/679 matrices...
Processed 210/679 matrices...
Processed 220/679 matrices...
Processed 230/679 matrices...
Processed 240/679 matrices...
Processed 250/679 matrices...
Processed 260/679 matrices...
Processed 270/679 matrices...
Processed 280/679 matrices...
Processed 290/679 matrices...
Processed 300/679 matrices...
Processed 310/679 matrices...
Processed 320/679 matrices...
Process

,count,mean,std,min,25%,50%,75%,max
edge_overlap_pct,679.0,31.397926,4.974581,21.212121,28.282828,30.303030,35.353535,46.464646
edge_jaccard_pct,679.0,18.727577,3.569020,11.864407,16.470588,17.857143,21.472393,30.263158
degree_l1,679.0,0.235317,0.066403,0.100000,0.180000,0.240000,0.280000,0.560000
avg_path_len,679.0,5.082779,1.012095,3.208991,4.379361,4.919298,5.554790,10.184199
avg_path_len_recon,679.0,4.466897,0.819780,2.940237,3.568059,4.747504,5.230644,5.640196
avg_path_len_diff,679.0,0.792090,0.594784,0.000866,0.376006,0.725219,1.036941,4.544003
avg_path_len_unweighted,679.0,6.022890,0.782204,4.483030,5.463131,5.975960,6.429596,10.100808
avg_path_len_unweighted_recon,679.0,5.004970,0.192633,4.374141,4.855758,5.081010,5.166566,5.187475
avg_path_len_unweighted_diff,679.0,1.082747,0.753094,0.007677,0.481616,1.031717,1.451111,5.101010
betweenness_topk_overlap_pct,679.0,43.284242,15.341546,10.000000,30.000000,40.000000,50.000000,90.000000



=== Processing val ===
Reconstruction error statistics (val dataset):


...

,mean,std,min,median,max
MSE,0.015776,0.004096,0.010304,0.014286,0.023460
MAE,0.097894,0.012649,0.079338,0.093987,0.120778
Frobenius,12.460835,1.584199,10.151062,11.952253,15.316725


Starting MST analysis for 146 matrices...
Processed 10/146 matrices...
Processed 20/146 matrices...
Processed 30/146 matrices...
Processed 40/146 matrices...
Processed 50/146 matrices...
Processed 60/146 matrices...
Processed 70/146 matrices...
Processed 80/146 matrices...
Processed 90/146 matrices...
Processed 100/146 matrices...
Processed 110/146 matrices...
Processed 120/146 matrices...
Processed 130/146 matrices...
Processed 140/146 matrices...
Saved metrics to: C:\Users\dylan\Desktop\TESI\SyntheticCorrelationVAE\PROGETTO_TESI_DYLAN\results\data_00_20_w252_s5\21_days_gap\reconstruction_analysis_test\AE_100dim_0001\reconstruction_errors_test.json

--- Summary Statistics (test dataset) ---


,count,mean,std,min,25%,50%,75%,max
edge_overlap_pct,146.0,24.816660,3.738094,16.161616,21.212121,25.252525,27.272727,32.323232
edge_jaccard_pct,146.0,14.217491,2.425370,8.791209,11.864407,14.450867,15.789474,19.277108
degree_l1,146.0,0.297808,0.046690,0.180000,0.260000,0.300000,0.320000,0.440000
avg_path_len,146.0,5.539962,1.460075,3.225454,3.822187,5.740255,6.650046,8.294018
avg_path_len_recon,146.0,4.366138,0.761146,3.173106,3.282370,4.722626,5.009421,5.240847
avg_path_len_diff,146.0,1.173824,0.885745,0.050609,0.409878,0.897435,1.812626,3.491625
avg_path_len_unweighted,146.0,7.040313,1.166003,5.291111,6.085202,6.662727,7.907323,10.050707
avg_path_len_unweighted_recon,146.0,5.055857,0.138203,4.825859,4.855758,5.146061,5.167475,5.187475
avg_path_len_unweighted_diff,146.0,1.984456,1.087132,0.435354,1.113434,1.633232,2.737475,4.885051
betweenness_topk_overlap_pct,146.0,14.452055,11.804781,0.000000,0.000000,20.000000,20.000000,40.000000



=== Processing all ===


FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\dylan\\Desktop\\TESI\\SyntheticCorrelationVAE\\PROGETTO_TESI_DYLAN\\models\\data_00_20_w252_s5\\21_days_gap\\AE\\AE_100dim_0001\\analysis_outputs\\all_reconstructed_AE_100dim_0001.pt'